# Парсинг, практика

In [35]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from time import sleep

## Задание 1

Дана ссылка на сайт [IMDb](https://www.imdb.com/calendar/?ref_=nv_mv_cal). Сохраните название и ссылки всех фильмов на странице. 

In [36]:
url = 'https://www.imdb.com/calendar/?ref_=nv_mv_cal'
page = requests.get(url,timeout=5)
page.status_code

ReadTimeout: HTTPSConnectionPool(host='www.imdb.com', port=443): Read timed out. (read timeout=5)

In [37]:
!pip install fake_useragent

In [38]:
from fake_useragent import UserAgent
UserAgent().chrome

'Mozilla/5.0 (Linux; Android 8.0; Pixel 2 Build/OPD3.170816.012) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/60.0.2693.1915 Mobile Safari/537.36'

In [39]:
page = requests.get(url, timeout=5,headers={'User-Agent': UserAgent().chrome})
page.status_code # 200 - все хорошо

ReadTimeout: HTTPSConnectionPool(host='www.imdb.com', port=443): Read timed out. (read timeout=5)

In [40]:
soup = BeautifulSoup(page.text)
if page.status_code != 200 or not soup.find_all('article'):
    print(f'IMDb не отдал страницу (код {page.status_code}) - читаем сохраненную копию imdb_calendar.html')
    with open('imdb_calendar.html', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read())

print(soup.prettify()[:2000]) # страница очень длинная, печатаем начало

IMDb не отдал страницу (код 202) - читаем сохраненную копию imdb_calendar.html
<!DOCTYPE html>
<html lang="en-US" xmlns:fb="http://www.facebook.com/2008/fbml" xmlns:og="http://opengraphprotocol.org/schema/">
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width" name="viewport"/>
  <script>
   if(typeof uet === 'function'){ uet('bb', 'LoadTitle', {wb: 1}); }
  </script>
  <script>
   window.addEventListener('load', (event) => {
        if (typeof window.csa !== 'undefined' && typeof window.csa === 'function') {
            var csaLatencyPlugin = window.csa('Content', {
                element: {
                    slotId: 'LoadTitle',
                    type: 'service-call'
                }
            });
            csaLatencyPlugin('mark', 'clickToBodyBegin', 1688025728045);
        }
    })
  </script>
  <title>
   Upcoming releases - IMDb
  </title>
  <meta content="IMDb is the world's most popular and authoritative source for movie, TV and celebrity content. Fi

In [41]:
dates = soup.find_all('h3')[1:]# все наши даты
dates

[<h3 class="ipc-title__text">
             Jun 29, 2023
            </h3>,
 <h3 class="ipc-title__text">
             Jul 06, 2023
            </h3>,
 <h3 class="ipc-title__text">
             Jul 13, 2023
            </h3>,
 <h3 class="ipc-title__text">
             Jul 15, 2023
            </h3>,
 <h3 class="ipc-title__text">
             Jul 20, 2023
            </h3>,
 <h3 class="ipc-title__text">
             Jul 27, 2023
            </h3>,
 <h3 class="ipc-title__text">
             Aug 03, 2023
            </h3>,
 <h3 class="ipc-title__text">
             Aug 10, 2023
            </h3>,
 <h3 class="ipc-title__text">
             Aug 11, 2023
            </h3>,
 <h3 class="ipc-title__text">
             Aug 17, 2023
            </h3>,
 <h3 class="ipc-title__text">
             Aug 24, 2023
            </h3>,
 <h3 class="ipc-title__text">
             Aug 31, 2023
            </h3>,
 <h3 class="ipc-title__text">
             Sep 07, 2023
            </h3>,
 <h3 class="ipc-title__te

In [42]:
len(dates)

39

In [45]:
dates[0].text.strip()

'Jun 29, 2023'

In [46]:
soup.find_all('h3')[1].text.strip()

'Jun 29, 2023'

In [47]:
imdb = soup.find_all('article')
len(imdb)

38

In [48]:
imdb[0]

<article class="sc-48add019-1 eovPBi" data-testid="calendar-section">
<div class="ipc-title ipc-title--base ipc-title--title ipc-title--on-textPrimary" data-testid="release-date">
<hgroup>
<h3 class="ipc-title__text">
            Jun 29, 2023
           </h3>
</hgroup>
</div>
<ul class="ipc-metadata-list ipc-metadata-list--dividers-after sc-48add019-2 hqwybd ipc-metadata-list--base" role="presentation">
<li class="ipc-metadata-list-summary-item ipc-metadata-list-summary-item--click sc-8c2b7f1f-0 bpqYIE" data-testid="coming-soon-entry">
<div class="ipc-poster ipc-poster--base sc-8c2b7f1f-1 knEssc ipc-sub-grid-item ipc-sub-grid-item--span-2" data-testid="poster" role="group">
<div class="ipc-media ipc-media--poster-27x40 ipc-image-media-ratio--poster-27x40 ipc-media--base ipc-media--custom ipc-poster__poster-image ipc-media__img" style="width:50px">
<img alt="Мэгги Мур(ы) (2023)" class="ipc-image" loading="lazy" sizes="50vw, (min-width: 480px) 34vw, (min-width: 600px) 26vw, (min-width: 1

In [49]:
imdb[0].find_all('a', {'class': 'ipc-metadata-list-summary-item__t'})

[<a aria-disabled="false" class="ipc-metadata-list-summary-item__t" href="/title/tt14181816/?ref_=rlm" role="button" tabindex="0">
               Мэгги Мур(ы) (2023)
              </a>,
 <a aria-disabled="false" class="ipc-metadata-list-summary-item__t" href="/title/tt13130024/?ref_=rlm" role="button" tabindex="0">
               Больше, чем люди (2023)
              </a>,
 <a aria-disabled="false" class="ipc-metadata-list-summary-item__t" href="/title/tt12579470/?ref_=rlm" role="button" tabindex="0">
               (Не)идеальные роботы (2023)
              </a>,
 <a aria-disabled="false" class="ipc-metadata-list-summary-item__t" href="/title/tt21391242/?ref_=rlm" role="button" tabindex="0">
               Время расплаты (2023)
              </a>,
 <a aria-disabled="false" class="ipc-metadata-list-summary-item__t" href="/title/tt18235306/?ref_=rlm" role="button" tabindex="0">
               Личная жизнь (2022)
              </a>,
 <a aria-disabled="false" class="ipc-metadata-list-summa

In [50]:
for num in range(0, len(dates)-1):
    print(dates[num].text.strip(), "\n")
    for i in imdb[num].find_all('a', {'class': 'ipc-metadata-list-summary-item__t'}):
        print(i.text.strip(), 'https://www.imdb.com'+i['href'])
    print('')

Jun 29, 2023 

Мэгги Мур(ы) (2023) https://www.imdb.com/title/tt14181816/?ref_=rlm
Больше, чем люди (2023) https://www.imdb.com/title/tt13130024/?ref_=rlm
(Не)идеальные роботы (2023) https://www.imdb.com/title/tt12579470/?ref_=rlm
Время расплаты (2023) https://www.imdb.com/title/tt21391242/?ref_=rlm
Личная жизнь (2022) https://www.imdb.com/title/tt18235306/?ref_=rlm
Восставшая из ада (2022) https://www.imdb.com/title/tt14617254/?ref_=rlm
Круиз по джунглям: Тайна Вальверде (2022) https://www.imdb.com/title/tt21249808/?ref_=rlm
Проклятие. Дом с прислугой 2 (2023) https://www.imdb.com/title/tt10211984/?ref_=rlm
Назад к динозаврам (2022) https://www.imdb.com/title/tt9672824/?ref_=rlm

Jul 06, 2023 

Гипнотик (2023) https://www.imdb.com/title/tt8080204/?ref_=rlm
Чарли и фантастическая четверка (2023) https://www.imdb.com/title/tt6349210/?ref_=rlm
В поисках снежного барса (2021) https://www.imdb.com/title/tt11937680/?ref_=rlm
Пантафа - пожирательница душ (2022) https://www.imdb.com/title/tt1

In [51]:
with open('imdb.txt', 'w', encoding='utf-8') as infile:
    for num in range(0, len(dates)-1):
        print(dates[num].text.strip(), "\n", file=infile)
        for i in imdb[num].find_all('a', {'class': 'ipc-metadata-list-summary-item__t'}):
            print(i.text.strip(), 'https://www.imdb.com'+i['href'], file=infile)
        print('', file=infile)

## Задание 2

Дана ссылка на первую страницу [каталога книжных новинок](https://www.biblio-globus.ru/catalog/group?id=444&page=1&sort=1&instock=). Соберите информацию о книгах со всех страниц.

*Примечание:* магазин переехал с bgshop.ru на biblio-globus.ru и поменял вёрстку, код ниже обновлён под новый сайт.

In [52]:
urls = []

for i in range(1,8):
    urls.append(f'https://www.biblio-globus.ru/catalog/group?id=444&page={i}&sort=1&instock=')
urls

['https://www.biblio-globus.ru/catalog/group?id=444&page=1&sort=1&instock=',
 'https://www.biblio-globus.ru/catalog/group?id=444&page=2&sort=1&instock=',
 'https://www.biblio-globus.ru/catalog/group?id=444&page=3&sort=1&instock=',
 'https://www.biblio-globus.ru/catalog/group?id=444&page=4&sort=1&instock=',
 'https://www.biblio-globus.ru/catalog/group?id=444&page=5&sort=1&instock=',
 'https://www.biblio-globus.ru/catalog/group?id=444&page=6&sort=1&instock=',
 'https://www.biblio-globus.ru/catalog/group?id=444&page=7&sort=1&instock=']

In [53]:
page = requests.get(urls[0], timeout=10)
page.encoding='utf-8'

In [54]:
soup = BeautifulSoup(page.text)
soup

<!DOCTYPE html>
<html prefix="og: http://ogp.me/ns#">
<head>
<meta charset="utf-8"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta content="fd9a658378927472" name="mailru-verification"/>
<title>Книжные новинки | Библио-Глобус</title>
<meta content="Еженедельное обновление новой литературы от крупных издательств. Будь в курсе самых популярных книжных новинок!" name="description"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<meta content="all,follow" name="robots"/>
<script src="https://www.google.com/recaptcha/api.js?render=6LeAtKsiAAAAAB2-39qRtUXN68LwgjmLeY5q8I33"></script>
<link href="/favicon.png" rel="icon" type="image/png"/>
<link href="/favicon.ico" rel="icon" type="image/x-icon"/>
<link href="/searchprovider.xml" rel="search" title="Biblio-Globus" type="application/opensearchdescription+xml"/>
<link href="https://static1.bgshop.ru" rel="dns-prefetch"/>
<link href="https://static2.bgshop.ru" rel="dns-prefetch"/>
<link href="https://mc.yand

In [60]:
# у каждой книги в каталоге есть картинка-ссылка с классом img-wrapper-index, она ведет на страницу книги
products = soup.find_all('a', {'class':'img-wrapper-index'})
print(len(products))
product = products[1]
product

12


<a class="img-wrapper-index" href="/product/11125838">
<img alt="Щелкунчик и Мышиный король" class="card-img" src="https://static1.bgshop.ru/imagehandler.ashx?fileName=11125838.jpg&amp;width=200"/>
</a>

In [61]:
# ссылка
url0 = 'https://www.biblio-globus.ru' + product.get('href')
url0

'https://www.biblio-globus.ru/product/11125838'

In [ ]:

page0 = requests.get(url0, timeout=10)
soup0 = BeautifulSoup(page0.text)
soup0

<!DOCTYPE html>
<html prefix="og: http://ogp.me/ns#">
<head>
<meta charset="utf-8"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta content="fd9a658378927472" name="mailru-verification"/>
<title>Книга «Щелкунчик и Мышиный король» Гофман Э. | ISBN 978-5-17-182195-1 | Библио-Глобус </title>
<meta content="Купить книгу «Щелкунчик и Мышиный король» Гофман Э. по низкой цене в книжном магазине «Библио-Глобус»." name="description"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<meta content="all,follow" name="robots"/>
<meta content="Щелкунчик и Мышиный король" property="og:title"/>
<meta content="book" property="og:type"/>
<meta content="Гофман Э. " property="og:book:author"/>
<meta content="«Щелкунчик и Мышиный король» — это атмосфера праздника, волшебства и тайны. Классическая история Э.Т.А. Гофмана про отважную Мари, непоседливого Фрица, смелого Щелкунчика и страшного Мышиного короля захватывает читателей вот уже..." property="og:description"/>
<meta

In [ ]:

main_info = soup0.find_all('div', {'id':'productMain'})[0]
main_info

<div class="row box" id="productMain">
<div class="col-sm-12 col-md-12 col-lg-3">
<a data-fancybox="gallery" href="https://static1.bgshop.ru/imagehandler.ashx?filename=11125838.jpg&amp;width=400">
<img alt="Книга «Щелкунчик и Мышиный король» Гофман Э." class="product_photo zoomIn img-fluid" src="https://static1.bgshop.ru/imagehandler.ashx?filename=11125838.jpg&amp;width=550" title="Щелкунчик и Мышиный король"/>
</a>
</div>
<div class="col-sm-12 col-md-12 col-lg-5">
<div class="item-desc">
<a class="badge bg-danger product-new" href="/catalog/group?id=444&amp;page=1&amp;sort=1">Новинка</a>
<a class="badge bg-danger product-bestseller" href="/catalog/group?id=535&amp;page=1&amp;sort=0&amp;inStock=on">Лидер продаж</a>
<div id="rating-wrapper">
<div class="starrr" data-target="#login-modal" data-toggle="modal">
<a aria-hidden="true" class="far fa-star" href="#"></a>
<a aria-hidden="true" class="far fa-star" href="#"></a>
<a aria-hidden="true" class="far fa-star" href="#"></a>
<a aria-hidde

In [66]:
import numpy as np

In [72]:
product = products[0]
url0 = 'https://www.biblio-globus.ru' + product.get('href')

page0 = requests.get(url0, timeout=10)
soup0 = BeautifulSoup(page0.text)
main_info = soup0.find_all('div', {'id':'productMain'})[0]
author = np.nan if main_info.find_all('p')[0].text.strip() == '' else main_info.find_all('p')[0].text.strip()
author

nan

In [73]:
title = main_info.find_all('h1')[0].text
title

'Александр Борисов.Служение Северу'

In [74]:
status = main_info.find_all('p', {'class':'item-status'})[0].text
status

'в наличии'

In [78]:
price = main_info.find_all('span', {'class':'price_new'})[0].text.split('\xa0₽')[0]
price

'7709'

In [79]:
# цена выглядит как '7709\xa0₽' - оставляем только цифры (и запятую, если есть копейки)
price_clean = float(re.sub(r'[^\d,]', '', price).replace(',', '.'))
price_clean

7709.0

In [83]:
from tqdm.notebook import tqdm
books = {}

for i in tqdm(soup.find_all('a', {'class':'img-wrapper-index'}),desc='books precessing'):
    url0 = 'https://www.biblio-globus.ru' + i.get('href')
    page0 = requests.get(url0, timeout=10)
    soup0 = BeautifulSoup(page0.text)
    sleep(1)

    main_info = soup0.find_all('div', {'id':'productMain'})[0]
    author = np.nan if main_info.find_all('p')[0].text.strip() == '' else main_info.find_all('p')[0].text.strip()# main_info.find_all('p')[0].text.strip()
    title = main_info.find_all('h1')[0].text.strip()

    status_tag = main_info.find('p', {'class':'item-status'})
    status = status_tag.text.strip() if status_tag else None

    price_tag = main_info.find('span', {'class':'price_new'}) # у книг не в наличии цены может не быть
    price_clean = float(re.sub(r'[^\d,]', '', price_tag.text).replace(',', '.')) if price_tag else None

    books[title] = [author, status, price_clean, url0]

books precessing:   0%|          | 0/12 [00:00<?, ?it/s]

In [84]:
books

{'Александр Борисов.Служение Северу': [nan,
  'в наличии',
  7709.0,
  'https://www.biblio-globus.ru/product/11125992'],
 'Щелкунчик и Мышиный король': ['Гофман Э.',
  'в наличии',
  1169.0,
  'https://www.biblio-globus.ru/product/11125838'],
 'Фокусы и трюки. Смотри и повторяй': ['Куанышбаев А., Мишина Д.А.',
  'в наличии',
  809.0,
  'https://www.biblio-globus.ru/product/11125835'],
 'Космос с Лунтиком Энциклопедия': ['Авсянникова Е.В.',
  'В наличии',
  1109.0,
  'https://www.biblio-globus.ru/product/11125804'],
 'Златовласка и три медведя в илл. Геннадия Спирина': ['Лихачев Д.А.',
  'В наличии',
  809.0,
  'https://www.biblio-globus.ru/product/11125798'],
 'Дыхание Кита': ['Боссу Р.',
  'в наличии',
  1409.0,
  'https://www.biblio-globus.ru/product/11125794'],
 'Восточные сказки. Рис. И. Петелиной': ['Салье М.А.',
  'в наличии',
  1279.0,
  'https://www.biblio-globus.ru/product/11125788'],
 'Азбука. Стихи, нейроигры, задания': ['Мельникова А. М.',
  'в наличии',
  899.0,
  'https:/

А теперь для всех страниц с книгами:

In [ ]:
# from tqdm import tqdm

In [ ]:
books = {}
for n_page,u in tqdm(enumerate(urls),desc='processing page'):
    page = requests.get(u, timeout=10)
    soup = BeautifulSoup(page.text)
    sleep(1)
    info = soup.find_all('a', {'class':'img-wrapper-index'})
    for i in tqdm(info,desc=f'processing books of page {n_page}'):
        url0 = 'https://www.biblio-globus.ru' + i.get('href')
        page0 = requests.get(url0, timeout=10)
        soup0 = BeautifulSoup(page0.text)
        sleep(1)

        main_info = soup0.find_all('div', {'id':'productMain'})[0]
        author = np.nan if main_info.find_all('p')[0].text.strip() == '' else main_info.find_all('p')[0].text.strip()# main_info.find_all('p')[0].text.strip()
        title = main_info.find_all('h1')[0].text.strip()

        status_tag = main_info.find('p', {'class':'item-status'})
        status = status_tag.text.strip() if status_tag else None

        price_tag = main_info.find('span', {'class':'price_new'}) # у книг не в наличии цены может не быть
        price_clean = float(re.sub(r'[^\d,]', '', price_tag.text).replace(',', '.')) if price_tag else None

        books[title] = [author, status, price_clean, url0]

processing page:   0%|          | 0/7 [00:00<?, ?it/s]

processing books:   0%|          | 0/12 [00:00<?, ?it/s]

processing books:   0%|          | 0/12 [00:00<?, ?it/s]

processing books:   0%|          | 0/12 [00:00<?, ?it/s]

processing books:   0%|          | 0/12 [00:00<?, ?it/s]

processing books:   0%|          | 0/12 [00:00<?, ?it/s]

processing books:   0%|          | 0/12 [00:00<?, ?it/s]

processing books:   0%|          | 0/12 [00:00<?, ?it/s]

In [181]:
df = pd.DataFrame(books, index = ['Автор', 'Статус', 'Цена', 'Ссылка']).transpose()
df
df = df.reset_index()

In [99]:
df.describe()

,index,Автор,Статус,Цена,Ссылка
count,84,84,84,84.0,84
unique,84,78,2,41.0,84
top,Александр Борисов.Служение Северу,Дж.К. Роулинг,в наличии,1079.0,https://www.biblio-globus.ru/product/11125992
freq,1,4,74,12.0,1


In [174]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   index   84 non-null     object
 1   Автор   84 non-null     object
 2   Статус  84 non-null     object
 3   Цена    84 non-null     object
 4   Ссылка  84 non-null     object
dtypes: object(5)
memory usage: 3.4+ KB


In [187]:
df.Автор = df.Автор.apply(lambda x: np.nan if x == '' else x)

In [104]:
df[df.Автор.isna()]

,index,Автор,Статус,Цена,Ссылка
0,Александр Борисов.Служение Северу,NaN,в наличии,7709.0,https://www.biblio-globus.ru/product/11125992
80,Программирование для детей,NaN,в наличии,1339.0,https://www.biblio-globus.ru/product/11124329


In [182]:
df

,index,Автор,Статус,Цена,Ссылка
0,Александр Борисов.Служение Северу,,в наличии,7709.0,https://www.biblio-globus.ru/product/11125992
1,Щелкунчик и Мышиный король,Гофман Э.,в наличии,1169.0,https://www.biblio-globus.ru/product/11125838
2,Фокусы и трюки. Смотри и повторяй,"Куанышбаев А., Мишина Д.А.",в наличии,809.0,https://www.biblio-globus.ru/product/11125835
3,Космос с Лунтиком Энциклопедия,Авсянникова Е.В.,В наличии,1109.0,https://www.biblio-globus.ru/product/11125804
4,Златовласка и три медведя в илл. Геннадия Спирина,Лихачев Д.А.,В наличии,809.0,https://www.biblio-globus.ru/product/11125798
...,...,...,...,...,...
79,Талисман,Кинг С.,в наличии,2459.0,https://www.biblio-globus.ru/product/11124342
80,Программирование для детей,,в наличии,1339.0,https://www.biblio-globus.ru/product/11124329
81,Тайная доктрина. Том 2. Антропогенезис. Золота...,Блаватская Е.П.,в наличии,1709.0,https://www.biblio-globus.ru/product/11124276
82,Тайная доктрина. Том 1. Космогенезис. Золотая ...,Блаватская Е.П.,в наличии,1709.0,https://www.biblio-globus.ru/product/11124275


In [ ]:
def get_author(name):
    author = np.nan
    if '.' in name:
        author = name.split('.')[0]
        if len(author.split(' '))!=2:
            author = np.nan
    return author
    

rename = df[df.Автор.isna()]['index'].apply(get_author)
rename

0     Александр Борисов
80                  NaN
Name: index, dtype: object

Решение проблемы Nan в одну строчку (на лекци были проблемы с тем что мы работали с копией датафрейма из-за чего наши изменения не вносились)


In [ ]:
mask = df['Автор'].isna()
df.loc[mask, 'Автор'] = df.loc[mask, 'index'].apply(get_author)
# по сути разница с тем что мы делали в лекции в том что мы полняем все за один прямой вызов df.loc[mask, 'Автор'], а как на лекции df.loc[mask].Автор, 
# тк во втрой реализаци мы берем копию столбца

In [185]:
df[df.Автор.isna()]

,index,Автор,Статус,Цена,Ссылка


In [128]:
list(rename.index)

[0, 80]

In [186]:
df

,index,Автор,Статус,Цена,Ссылка
0,Александр Борисов.Служение Северу,,в наличии,7709.0,https://www.biblio-globus.ru/product/11125992
1,Щелкунчик и Мышиный король,Гофман Э.,в наличии,1169.0,https://www.biblio-globus.ru/product/11125838
2,Фокусы и трюки. Смотри и повторяй,"Куанышбаев А., Мишина Д.А.",в наличии,809.0,https://www.biblio-globus.ru/product/11125835
3,Космос с Лунтиком Энциклопедия,Авсянникова Е.В.,В наличии,1109.0,https://www.biblio-globus.ru/product/11125804
4,Златовласка и три медведя в илл. Геннадия Спирина,Лихачев Д.А.,В наличии,809.0,https://www.biblio-globus.ru/product/11125798
...,...,...,...,...,...
79,Талисман,Кинг С.,в наличии,2459.0,https://www.biblio-globus.ru/product/11124342
80,Программирование для детей,,в наличии,1339.0,https://www.biblio-globus.ru/product/11124329
81,Тайная доктрина. Том 2. Антропогенезис. Золота...,Блаватская Е.П.,в наличии,1709.0,https://www.biblio-globus.ru/product/11124276
82,Тайная доктрина. Том 1. Космогенезис. Золотая ...,Блаватская Е.П.,в наличии,1709.0,https://www.biblio-globus.ru/product/11124275


In [161]:
rename = pd.DataFrame(rename)
rename.columns = ['Автор']
rename

,Автор
0,Александр Борисов
80,NaN


In [169]:
rename

,Автор
0,Александр Борисов
80,NaN


In [171]:
result = pd.concat([rename,df[~df.Автор.isna()]], axis = 1)


In [172]:
result

,Автор,index,Автор,Статус,Цена,Ссылка,"(0, Автор)","(80, Автор)"
0,Александр Борисов,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Щелкунчик и Мышиный король,Гофман Э.,в наличии,1169.0,https://www.biblio-globus.ru/product/11125838,Александр Борисов,NaN
2,NaN,Фокусы и трюки. Смотри и повторяй,"Куанышбаев А., Мишина Д.А.",в наличии,809.0,https://www.biblio-globus.ru/product/11125835,Александр Борисов,NaN
3,NaN,Космос с Лунтиком Энциклопедия,Авсянникова Е.В.,В наличии,1109.0,https://www.biblio-globus.ru/product/11125804,Александр Борисов,NaN
...,...,...,...,...,...,...,...,...
78,NaN,Технолайк. Энциклопедия о цифровой безопасности,"Соколенко А.М., Соколенко В.А.",в наличии,1169.0,https://www.biblio-globus.ru/product/11124346,Александр Борисов,NaN
79,NaN,Талисман,Кинг С.,в наличии,2459.0,https://www.biblio-globus.ru/product/11124342,Александр Борисов,NaN
81,NaN,Тайная доктрина. Том 2. Антропогенезис. Золота...,Блаватская Е.П.,в наличии,1709.0,https://www.biblio-globus.ru/product/11124276,Александр Борисов,NaN
82,NaN,Тайная доктрина. Том 1. Космогенезис. Золотая ...,Блаватская Е.П.,в наличии,1709.0,https://www.biblio-globus.ru/product/11124275,Александр Борисов,NaN


In [ ]:
for item in rename.items():
    df[item[0],'Автор'] = item[1]

In [ ]:
df = df.fillna()

In [135]:
df.iloc[list(rename.index)].Автор = rename

/var/folders/9p/5zv6x_w96ln02yhb9csv96t96rqzcv/T/ipykernel_9596/2207978694.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[list(rename.index)].Автор = rename


In [ ]:
df.fillna()

In [144]:
df[df.Автор.isna()]

,index,Автор,Статус,Цена,Ссылка,"(0, Автор)","(80, Автор)"
0,Александр Борисов.Служение Северу,NaN,в наличии,7709.0,https://www.biblio-globus.ru/product/11125992,Александр Борисов,NaN
80,Программирование для детей,NaN,в наличии,1339.0,https://www.biblio-globus.ru/product/11124329,Александр Борисов,NaN


In [115]:
df[df.Автор.isna()]

,index,Автор,Статус,Цена,Ссылка
0,Александр Борисов.Служение Северу,NaN,в наличии,7709.0,https://www.biblio-globus.ru/product/11125992
80,Программирование для детей,NaN,в наличии,1339.0,https://www.biblio-globus.ru/product/11124329


In [170]:
df.to_excel('bg_books.xlsx')